# Module 16: Linear Transformations and Matrices over GF(2)

**Volume 1: Foundations, Finite Fields, and AES**

This notebook builds matrix-vector multiplication over GF(2) from scratch and explores
every concept from the tutorial through runnable code.

**Sections**
1. GF(2) arithmetic basics
2. Vectors and matrices as Python lists
3. Matrix-vector multiplication
4. Identity and permutation matrices
5. Mixing matrices
6. Linear vs affine transformations
7. The AES affine matrix
8. Verifying the linearity property for all inputs
9. Invertibility explorer
10. Summary table and bridge to Module 17

## Section 1: GF(2) Arithmetic Basics

Over GF(2), the only values are 0 and 1.  Addition is XOR (no carrying).  Multiplication is AND.

| a | b | a + b (XOR) | a · b (AND) |
|---|---|-------------|-------------|
| 0 | 0 |      0      |      0      |
| 0 | 1 |      1      |      0      |
| 1 | 0 |      1      |      0      |
| 1 | 1 |      0      |      1      |

The shortcut: XOR a sequence of bits and you get 1 if an **odd** number of them are 1.

In [ ]:
# GF(2) arithmetic
def gf2_add(a, b):
    """Addition in GF(2) = XOR."""
    return a ^ b

def gf2_mul(a, b):
    """Multiplication in GF(2) = AND."""
    return a & b

def gf2_xor_chain(*bits):
    """XOR an arbitrary sequence of bits."""
    result = 0
    for b in bits:
        result ^= b
    return result

# Demonstrate
print('GF(2) addition (XOR) table:')
for a in [0, 1]:
    for b in [0, 1]:
        print(f'  {a} + {b} = {gf2_add(a,b)}')

print()
print('XOR chain examples:')
print('1 + 0 + 1 =', gf2_xor_chain(1, 0, 1))   # 0 (even number of 1s)
print('1 + 1 + 1 =', gf2_xor_chain(1, 1, 1))   # 1 (odd number of 1s)
print('1 + 1 + 0 + 1 =', gf2_xor_chain(1, 1, 0, 1))  # 1

## Section 2: Vectors and Matrices as Python Lists

A **vector** over GF(2) is an ordered list of bits.  A **matrix** is a list of rows, where each row is a list of bits.

We store matrices in row-major order: `A[i]` is row *i*.

```
A = [[1, 1, 0, 0],
     [0, 1, 1, 0],
     [0, 0, 1, 1],
     [1, 0, 0, 1]]
```

Row *i* is the **recipe** for output bit *y_i*: a 1 means "include that input bit", a 0 means "ignore it".

In [ ]:
def print_matrix(A, name='A'):
    """Pretty-print a binary matrix."""
    print(f'{name} =')
    for row in A:
        print(' ', row)

def print_vector(v, name='v'):
    """Pretty-print a bit vector."""
    print(f'{name} = {v}')

# The mixing matrix M from the tutorial
M = [[1, 1, 0, 0],
     [0, 1, 1, 0],
     [0, 0, 1, 1],
     [1, 0, 0, 1]]

v = [1, 0, 1, 1]

print_matrix(M)
print()
print_vector(v)

## Section 3: Matrix-Vector Multiplication

To compute **y = Av**:

1. For each row *i* of A, find the positions *j* where `A[i][j] == 1`.
2. XOR the input bits `v[j]` at those positions.
3. The result is output bit `y[i]`.

This is the core operation.  Each row is processed independently.

In [ ]:
def gf2_matvec(A, v):
    """
    Matrix-vector multiplication over GF(2).
    A: list of n rows, each a list of n bits.
    v: list of n bits.
    Returns: y = Av as a list of n bits.
    """
    n = len(A)
    y = []
    for i in range(n):
        bit = 0
        for j in range(n):
            bit ^= A[i][j] * v[j]   # include v[j] only if A[i][j] == 1
        y.append(bit)
    return y

def gf2_matvec_verbose(A, v):
    """Same as gf2_matvec but prints each row computation."""
    n = len(A)
    y = []
    for i in range(n):
        terms = [f'v{j}({v[j]})' for j in range(n) if A[i][j] == 1]
        if not terms:
            terms = ['0']
        formula = ' XOR '.join(terms)
        bit = 0
        for j in range(n):
            bit ^= A[i][j] * v[j]
        print(f'  y{i} = {formula} = {bit}')
        y.append(bit)
    return y

print('M * v (verbose):')
y = gf2_matvec_verbose(M, v)
print()
print('Result y =', y)

## Section 4: Identity and Permutation Matrices

**Identity matrix I**: row *i* has a single 1 at position *i*.  Effect: `Iv = v` (pass-through).

**Permutation matrix P**: each row has exactly one 1.  Effect: rearranges (permutes) the bit positions.

Both are special cases of linear transformations.  Both are invertible.

In [ ]:
# 4x4 identity
I4 = [[1 if i==j else 0 for j in range(4)] for i in range(4)]

# 4x4 reversal permutation (sends position 0 -> 3, 1 -> 2, 2 -> 1, 3 -> 0)
P_reverse = [[0,0,0,1],[0,0,1,0],[0,1,0,0],[1,0,0,0]]

v_test = [1, 0, 1, 1]

print('Identity matrix I4:')
print_matrix(I4, 'I4')
print()
print('I4 * v =', gf2_matvec(I4, v_test), '  (should equal v =', v_test, ')')

print()
print('Reversal permutation P_reverse:')
print_matrix(P_reverse, 'P')
print()
print('P * v =', gf2_matvec(P_reverse, v_test), '  (should be reversed =', v_test[::-1], ')')

## Section 5: Mixing Matrices

A **mixing matrix** has multiple 1s per row.  Each output bit depends on multiple input bits (XOR of two or more).  This creates **diffusion**: flipping one input bit changes multiple output bits.

Mixing matrices are the algebraic description of diffusion layers in block ciphers.

In [ ]:
def hamming_weight(v):
    """Count the number of 1s in a bit vector."""
    return sum(v)

def flip_bit(v, pos):
    """Return a copy of v with bit at position pos flipped."""
    w = v[:]
    w[pos] ^= 1
    return w

def count_changed_bits(y1, y2):
    """Count positions where y1 and y2 differ."""
    return sum(a ^ b for a, b in zip(y1, y2))

# Show diffusion: flipping each input bit of v and counting how many output bits change
v_base = [1, 0, 1, 1]
y_base = gf2_matvec(M, v_base)

print('Mixing matrix M diffusion test:')
print('Base v    =', v_base, '  ->  y =', y_base)
print()
for pos in range(4):
    v_flip = flip_bit(v_base, pos)
    y_flip = gf2_matvec(M, v_flip)
    changed = count_changed_bits(y_base, y_flip)
    print(f'  Flip bit {pos}: v={v_flip} -> y={y_flip}  ({changed} output bit(s) changed)')

print()
print('Compare with identity (no mixing):')
y_id = gf2_matvec(I4, v_base)
for pos in range(4):
    v_flip = flip_bit(v_base, pos)
    y_flip = gf2_matvec(I4, v_flip)
    changed = count_changed_bits(y_id, y_flip)
    print(f'  Flip bit {pos}: y={y_flip}  ({changed} output bit changed)')

## Section 6: Linear vs Affine Transformations

**Linear:**  `y = A * v`

**Affine:**  `y = A * v XOR c`  (add a fixed constant vector c)

The difference is that an affine transformation has a **bias** — the constant shifts the output away from zero when the input is zero.

```
Linear:  L(0) = A*0 = 0       (zero maps to zero)
Affine:  F(0) = A*0 XOR c = c (zero maps to c, not zero)
```

This distinction matters for cryptanalysis: purely linear ciphers leak algebraic structure.

In [ ]:
def gf2_affine(A, v, c):
    """
    Affine transformation over GF(2): y = Av XOR c
    A: matrix, v: input vector, c: constant vector (same length as v)
    """
    Av = gf2_matvec(A, v)
    return [Av[i] ^ c[i] for i in range(len(c))]

# Use mixing matrix M and a constant c
c = [1, 1, 0, 0]
v_zero = [0, 0, 0, 0]
v_test = [1, 0, 1, 1]

print('Linear transformation y = Mv:')
print('  M * 0000 =', gf2_matvec(M, v_zero),  '  (zero in -> zero out)')
print('  M * 1011 =', gf2_matvec(M, v_test))

print()
print('Affine transformation y = Mv XOR c, c =', c)
print('  F(0000) =', gf2_affine(M, v_zero, c), '  (zero in -> c =', c, ')')
print('  F(1011) =', gf2_affine(M, v_test, c))

print()
print('Key difference: L(0) = 0 for linear; F(0) = c for affine')

## Section 7: The AES Affine Matrix

The affine layer of the AES S-box uses a specific 8×8 **circulant matrix** A together with
constant c = 0x63 = [1,1,0,0,0,1,1,0] (LSB first).

The formula is:
```
s_i = b_i XOR b_{(i+4)%8} XOR b_{(i+5)%8} XOR b_{(i+6)%8} XOR b_{(i+7)%8} XOR c_i
```

This is exactly `s = Ab XOR c` where A encodes those four rotational shifts.

In [ ]:
# The AES affine matrix (8x8 circulant)
AES_A = [
    [1,0,0,0,1,1,1,1],
    [1,1,0,0,0,1,1,1],
    [1,1,1,0,0,0,1,1],
    [1,1,1,1,0,0,0,1],
    [1,1,1,1,1,0,0,0],
    [0,1,1,1,1,1,0,0],
    [0,0,1,1,1,1,1,0],
    [0,0,0,1,1,1,1,1]
]

# AES constant c = 0x63 in LSB-first bit order
# 0x63 = 0110 0011 binary; LSB first = [1,1,0,0,0,1,1,0]
AES_C = [1, 1, 0, 0, 0, 1, 1, 0]

def int_to_bits_lsb(x, n=8):
    """Convert integer to n-bit LSB-first list."""
    return [(x >> i) & 1 for i in range(n)]

def bits_lsb_to_int(bits):
    """Convert LSB-first bit list to integer."""
    return sum(b << i for i, b in enumerate(bits))

print('AES affine matrix A (8x8):')
print_matrix(AES_A, 'AES_A')
print()
print('AES constant c = 0x63 (LSB first):', AES_C)

# Apply to example input 0x53 (b = bits of field inverse 0xCA)
# From Module 14: gfInv(0x53) = 0xCA
b_CA = int_to_bits_lsb(0xCA)  # bits of 0xCA LSB first
s_bits = gf2_affine(AES_A, b_CA, AES_C)
s_byte = bits_lsb_to_int(s_bits)

print()
print(f'Example: input byte 0x53, field-inverse = 0xCA')
print(f'b (0xCA bits, LSB first) = {b_CA}')
print(f'Affine output bits (LSB first) = {s_bits}')
print(f'Affine output byte = 0x{s_byte:02X}  (expected: 0xED from the AES S-box)')

## Section 8: Verifying the Linearity Property for All Inputs

A transformation L is linear if and only if:
```
L(u XOR v) = L(u) XOR L(v)    for ALL u, v
```

We can verify this exhaustively for all 256 pairs of 4-bit vectors (or all 65536 pairs for 8-bit).
Every matrix multiplication over GF(2) must pass this test.

In [ ]:
def verify_linearity(A, bits=4):
    """
    Verify L(u XOR v) = L(u) XOR L(v) for all pairs of bit vectors.
    Returns (True, checked_count) if linearity holds for all pairs.
    """
    n = 2 ** bits
    failures = 0
    checked = 0

    for ui in range(n):
        u = int_to_bits_lsb(ui, bits)
        Lu = gf2_matvec(A, u)
        for vi in range(n):
            v = int_to_bits_lsb(vi, bits)
            Lv = gf2_matvec(A, v)
            # Compute u XOR v
            uv = [u[j] ^ v[j] for j in range(bits)]
            Luv = gf2_matvec(A, uv)
            # Check L(u) XOR L(v)
            LuXorLv = [Lu[j] ^ Lv[j] for j in range(bits)]
            if Luv != LuXorLv:
                failures += 1
            checked += 1

    return failures == 0, checked, failures

# Verify mixing matrix M
ok, checked, fails = verify_linearity(M, bits=4)
print(f'Mixing matrix M: linearity holds = {ok} (checked {checked} pairs, {fails} failures)')

# Verify identity
ok, checked, fails = verify_linearity(I4, bits=4)
print(f'Identity matrix I4: linearity holds = {ok} (checked {checked} pairs, {fails} failures)')

# Verify AES affine matrix (8-bit)
ok, checked, fails = verify_linearity(AES_A, bits=8)
print(f'AES affine matrix A: linearity holds = {ok} (checked {checked} pairs, {fails} failures)')

# Demonstrate that the AFFINE transformation (with constant) is NOT linear
print()
print('Affine transformation F(v) = Mv XOR c is NOT linear:')
u_test = [1, 0, 0, 0]
v_test = [0, 1, 0, 0]
uv_test = [u_test[j] ^ v_test[j] for j in range(4)]
Fu = gf2_affine(M, u_test, c)
Fv = gf2_affine(M, v_test, c)
Fuv = gf2_affine(M, uv_test, c)
FuXFv = [Fu[j] ^ Fv[j] for j in range(4)]
print(f'  F(u XOR v) = {Fuv}')
print(f'  F(u) XOR F(v) = {FuXFv}')
print(f'  Equal: {Fuv == FuXFv}  <-- False: affine is not linear')

## Section 9: Invertibility Explorer

A matrix A over GF(2) is invertible if there exists a matrix A⁻¹ such that A⁻¹A = I.

We can test invertibility by checking whether the transformation is **bijective** — every output vector is produced by exactly one input vector.  If any two inputs produce the same output, the matrix is not invertible.

For 4-bit vectors: iterate over all 16 possible inputs and collect the outputs.  If all 16 outputs are distinct, the matrix is invertible.

In [ ]:
def is_invertible(A, bits=4):
    """
    Test if A is invertible over GF(2) by checking bijectivity.
    Returns True if all outputs are distinct.
    """
    n = 2 ** bits
    outputs = set()
    for i in range(n):
        v = int_to_bits_lsb(i, bits)
        y = gf2_matvec(A, v)
        outputs.add(tuple(y))
    return len(outputs) == n

def find_inverse(A, bits=4):
    """
    Find A^{-1} by brute force over all 2^(n^2) binary matrices.
    Practical only for small n (n=4 means 2^16 = 65536 candidates).
    Returns the inverse matrix or None if not invertible.
    """
    n = bits
    total = 2 ** (n * n)
    for code in range(total):
        # Decode candidate matrix
        B = []
        tmp = code
        for i in range(n):
            row = []
            for j in range(n):
                row.append(tmp & 1)
                tmp >>= 1
            B.append(row)
        # Check BA = I
        is_inv = True
        for col in range(n):
            e = [1 if j == col else 0 for j in range(n)]  # standard basis vector
            Ae = gf2_matvec(A, e)    # column col of A
            BAe = gf2_matvec(B, Ae)
            if BAe != e:
                is_inv = False
                break
        if is_inv:
            return B
    return None

# Test matrices
matrices = [
    ('Identity I4',   I4),
    ('Permutation P', P_reverse),
    ('Mixing M',      M),
    ('Singular S',    [[1,1,0,0],[1,1,0,0],[0,0,1,0],[0,0,0,1]]),  # two identical rows
]

for name, A in matrices:
    inv = is_invertible(A, bits=4)
    print(f'{name:20s}: invertible = {inv}')

print()
print('Finding inverse of mixing matrix M (brute force over 4x4 matrices)...')
M_inv = find_inverse(M, bits=4)
if M_inv:
    print('M^{-1} found:')
    print_matrix(M_inv, 'M_inv')
    # Verify: apply M then M_inv to a test vector
    v_verify = [1, 0, 1, 1]
    y = gf2_matvec(M, v_verify)
    recovered = gf2_matvec(M_inv, y)
    print()
    print(f'Verify: M(v)={y}, M_inv(M(v))={recovered}, original v={v_verify}')
    print(f'Round-trip correct: {recovered == v_verify}')

## Section 10: Summary Table and Bridge to Module 17

### Key concepts from Module 16

| Concept | Plain meaning | Key property |
|---------|---------------|-------------|
| GF(2) arithmetic | XOR-based, only 0 and 1 | 1+1=0, no carrying |
| Matrix row | Bit recipe for one output | 1 = include, 0 = ignore |
| y = Av | Matrix-vector multiply | Apply each row independently |
| Linear: y = Av | No constant | L(0)=0, L(u⊕v)=L(u)⊕L(v) |
| Affine: y = Av⊕c | Plus a constant | F(0)=c, not linear |
| Invertible matrix | Round-trip possible | A⁻¹A = I |
| AES affine | 8×8 circulant + 0x63 | Links Module 14 to this module |

### Bridge to Module 17

**Module 17: Diffusion, Branch Number, and Avalanche** takes the matrix framework from this module
and asks a more refined question: _how good_ is a given mixing matrix at spreading changes?

The **branch number** of a matrix is the minimum number of positions that change in input + output
when any single input bit is flipped.  A matrix with high branch number provides strong diffusion.

The **MixColumns** matrix in AES achieves the maximum possible branch number of 5 (for 4×4 over GF(2⁸)),
which is why it is such an effective diffusion layer.

In [ ]:
# Preview: computing a simple branch number for 4-bit matrices
def branch_number_4bit(A):
    """
    Compute the branch number of a 4x4 GF(2) matrix.
    Branch number = min over all nonzero v of: hw(v) + hw(Av)
    where hw = Hamming weight (number of 1s).
    """
    n_bits = 4
    min_bn = n_bits * 2 + 1  # start high
    for i in range(1, 2**n_bits):   # skip zero vector
        v = int_to_bits_lsb(i, n_bits)
        Av = gf2_matvec(A, v)
        bn = hamming_weight(v) + hamming_weight(Av)
        if bn < min_bn:
            min_bn = bn
    return min_bn

print('Branch numbers (preview of Module 17):')
for name, mat in [('Identity I4', I4), ('Permutation P', P_reverse), ('Mixing M', M)]:
    if is_invertible(mat, bits=4):  # only defined for invertible matrices
        bn = branch_number_4bit(mat)
        print(f'  {name:20s}: branch number = {bn}')
    else:
        print(f'  {name:20s}: not invertible — branch number undefined')

print()
print('The identity has branch number 2 (a single flipped bit affects only one output bit).')
print('A good mixing matrix should have a higher branch number — more diffusion.')